# Data Preparation

### Imports and constants

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# shared data-acquisition functions live in scripts/download.py
# (paths there are computed from the module location, so cwd doesn't matter)
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from download import (
    RAW_DIR, PROCESSED_DIR, CDR_PATH,
    download_cdr, download_gdc_manifest, download_gdc_files, resolve_barcodes,
)

COHORTS = ["LUSC"]

### Download utilities

The GDC/TCGA-CDR acquisition functions (`download_cdr`, `download_gdc_manifest`, `download_gdc_files`, `resolve_barcodes`) live in `scripts/download.py` and are imported above — the same module is reused by `scripts/run_cohort.py` so the download logic has a single source of truth.

# Data download and initial filters

### Step 1. Joining the gene expression file and the aliquot barcodes

The TCGA RNA-seq TSVs don't carry the patient barcode in the file body. We query the GDC `/files` endpoint for every file UUID in the manifest and pull back the nested aliquot barcode. 

| column | meaning |
|---|---|
| `file_id` | GDC file UUID — matches the folder name in `data/raw/{COHORT}/` |
| `patient_barcode` | `TCGA-XX-XXXX` — join key to TCGA-CDR |
| `sample_barcode` | `TCGA-XX-XXXX-01A` — adds sample-type + vial |
| `aliquot_barcode` | full 28-char barcode — the canonical id for the row |
| `sample_type` | should all be `Primary Tumor` (we filtered the manifest on this) |
| `tss` | tissue source site code (positions 2-3 of the patient barcode) — the institution |

### Step 2. Deduplicating aliquots

Some patients have >1 aliquot sequenced (different vials, repeat extractions, multiple sequencing centres). Each aliquot is a separate file in GDC but the survival label is patient-level — so duplicate aliquots would upweight a non-random subset of patients in the likelihood.

**Rule:** keep the alphabetically first `aliquot_barcode` per patient. 


### Step 3. Join TCGA-CDR survival labels

Source: `data/raw/tcga_cdr.xlsx`, sheet `TCGA-CDR` (Liu et al. 2018). Join on `bcr_patient_barcode == patient_barcode`.

### Step 4. Build the gene matrix

For each surviving patient, read the gene counts, take the `fpkm_uq_unstranded` keyed by `gene_id` and stack into a (patients x genes) matrix

In [ ]:
# Download the common tcga cdr file
download_cdr()

for cohort in COHORTS:
    # Download the data
    print(f"Downloading GDC manifest and files for cohort: {cohort}")
    download_gdc_manifest(cohort)
    download_gdc_files(cohort)
    barcodes = resolve_barcodes(cohort)

    # Checking if barcodes are correct
    print("barcodes df shape:", barcodes.shape)
    assert (barcodes["sample_type"] == "Primary Tumor").all()
    assert (barcodes["project_id"] == f"TCGA-{cohort}").all()

    # Deduplicating aliquots
    dedup = (
            barcodes.sort_values("aliquot_barcode")
            .groupby("patient_barcode", as_index=False)
            .first()
        )

    print(f"\nbefore: {len(barcodes)} files")
    print(f"after:  {len(dedup)} patients")
    print(f"dropped {len(barcodes) - len(dedup)} duplicate aliquots")


    # Joining survival labels
    tcga_cdr = pd.read_excel(RAW_DIR / "tcga_cdr.xlsx", sheet_name="TCGA-CDR")
    cohort_cdr = tcga_cdr[tcga_cdr["type"] == cohort].copy()

    n_matched = dedup["patient_barcode"].isin(cohort_cdr["bcr_patient_barcode"]).sum()
    print(f"patients with survival labels: {n_matched} / {len(dedup)} "
          f"(dropped {len(dedup) - n_matched} with missing/no survival data)")

    # Building the gene matrix
    gene_matrix = pd.DataFrame()

    for file_id in dedup["file_id"]:
        tsv = next(Path(f"{RAW_DIR}/{cohort}/{file_id}").glob("*.rna_seq.augmented_star_gene_counts.tsv"))
        df = pd.read_csv(tsv, sep="\t", skiprows=1)
        df = df[df["gene_id"].str.startswith("ENSG")]
        df = df[df["gene_type"] == "protein_coding"] 
        df = df[["gene_id", "fpkm_uq_unstranded"]]
        df.set_index("gene_id", inplace=True)
        df.columns = [file_id]
        gene_matrix = pd.concat([gene_matrix, df], axis=1)

    gene_matrix.to_csv(f"{RAW_DIR}/{cohort}/gene_matrix.csv")

TCGA-CDR already present at /Users/affanhamid/projects/tcga-survival-bias/data/raw/tcga_cdr.xlsx, skipping.
Manifest already present at /Users/affanhamid/projects/tcga-survival-bias/data/raw/LUSC/manifest.txt, skipping.
All 511 LUSC files already downloaded, skipping.
Barcodes already present at /Users/affanhamid/projects/tcga-survival-bias/data/raw/LUSC/barcodes.csv, skipping.
barcodes df shape: (511, 8)

before: 511 files
after:  501 patients
dropped 10 duplicate aliquots
patients with survival labels: 495 / 501 (dropped 6 with missing/no survival data)


# Data Preprocessing

## Gene filter, transform, standardise

- Keep genes with FPKM-UQ at least 1 in at least 10% of the patients
- Apply `log2(FPKM_UQ + 1)`.
- Standardise each gene to mean 0, std 1 across patients **within this cohort**.

In [ ]:
cohort = COHORTS[0]
gene_matrix = pd.read_csv(f"{RAW_DIR}/{cohort}/gene_matrix.csv", index_col="gene_id")

# expression filter: keep genes with FPKM-UQ >= 1 in >= 10% of patients
# also removes all-zero / zero-variance genes that would NaN on standardisation
n = gene_matrix.shape[1]
keep = (gene_matrix >= 1).sum(axis=1) >= 0.1 * n
gene_matrix = gene_matrix.loc[keep]
print("genes after expression filter:", gene_matrix.shape[0])

# log2(FPKM-UQ + 1)
logm = np.log2(gene_matrix + 1)

# standardise each gene across patients (mean 0, std 1)
zscore = logm.sub(logm.mean(axis=1), axis=0).div(logm.std(axis=1), axis=0)
assert not zscore.isna().any().any(), "NaNs in zscore — a zero-variance gene slipped through"

genes after expression filter: 13859


In [ ]:

# 1. transpose -> patients (file_id) x genes
X = zscore.T 
X.index.name = "file_id"

# 2. attach patient_barcode + tss from dedup
meta = dedup.set_index("file_id")[["patient_barcode", "tss"]]
X = meta.join(X)                       # rows still keyed by file_id

# 3. join survival labels from TCGA-CDR (all endpoints; NaNs kept by design)
labels = (
    cohort_cdr[["bcr_patient_barcode", "OS", "OS.time", "DSS", "DSS.time", "PFI", "PFI.time", "DFI", "DFI.time"]]
    .rename(columns={
        "bcr_patient_barcode": "patient_barcode",
        "OS.time": "OS_time",
        "DSS.time": "DSS_time",
        "PFI.time": "PFI_time",
        "DFI.time": "DFI_time"
    })
)
final = X.merge(labels, on="patient_barcode", how="inner")
print(f"patients: {len(X)} gene-matrix -> {len(final)} after survival join "
      f"(dropped {len(X) - len(final)})")

# 4. order columns: ids, then endpoint labels, then genes
id_cols    = ["patient_barcode", "tss"]
label_cols = ["OS", "OS_time", "DSS", "DSS_time", "PFI", "PFI_time", "DFI", "DFI_time"]
gene_cols  = [c for c in final.columns if c.startswith("ENSG")]
final = final[id_cols + label_cols + gene_cols]

# 5. sanity checks — ids + genes must be complete; endpoint labels may be missing by design
assert final["patient_barcode"].is_unique, "duplicate patients in final table"
assert not final[id_cols + gene_cols].isna().any().any(), "NaNs in ids or gene matrix"
print("missing per endpoint:\n", final[label_cols].isna().sum())

# 6. write -> per-cohort folder data/processed/{cohort}/{cohort}.parquet
cohort_out = PROCESSED_DIR / cohort
cohort_out.mkdir(parents=True, exist_ok=True)
final.to_parquet(cohort_out / f"{cohort}.parquet", index=False)
print("final table:", final.shape, "->", cohort_out / f"{cohort}.parquet")